### Exo-MerCat example

In [1]:
from astropy.table import Table


input_table = Table.read("./input/HPIC_LC4_combined_d50.txt", format="ascii")

In [2]:
from crossmatching import EMCCatalog, EMCIdSupplier, Crossmatcher

cme = Crossmatcher(EMCCatalog(), EMCIdSupplier())
cme.load_catalog(from_file="./exo-mercat.csv")
cme.load_alternate_ids(input_table["star_name"], from_file="./exo-mercat.csv")

emc_result = cme.combined_crossmatch(input_table, input_starname_key="star_name")

In [3]:
from collections import Counter
Counter(emc_result["match_type"])

Counter({np.str_('id+coordinates'): 1281, np.str_('coordinates'): 11})

In [4]:
emc_result.colnames

['star_name',
 'sy_dist',
 'st_spectype',
 'st_rad',
 'st_teff',
 'st_mass',
 'st_age',
 'ra',
 'dec',
 'sy_vmag',
 'sy_jmag',
 'sy_hmag',
 'sy_kmag',
 'known_binary_fl',
 'gaia_binary_fl',
 'WDSsep',
 'wds_deltamag',
 'exo-mercat_name',
 'nasa_name',
 'toi_name',
 'epic_name',
 'eu_name',
 'oec_name',
 'host',
 'letter',
 'main_id',
 'binary',
 'main_id_ra',
 'main_id_dec',
 'mass',
 'mass_max',
 'mass_min',
 'mass_url',
 'msini',
 'msini_max',
 'msini_min',
 'msini_url',
 'bestmass',
 'bestmass_max',
 'bestmass_min',
 'bestmass_url',
 'bestmass_provenance',
 'p',
 'p_max',
 'p_min',
 'p_url',
 'r',
 'r_max',
 'r_min',
 'r_url',
 'a',
 'a_max',
 'a_min',
 'a_url',
 'e',
 'e_max',
 'e_min',
 'e_url',
 'i',
 'i_max',
 'i_min',
 'i_url',
 'discovery_method',
 'status',
 'checked_status_string',
 'original_status_string',
 'confirmed',
 'discovery_year',
 'main_id_aliases',
 'main_id_provenance',
 'angular_separation_flag',
 'angular_separation',
 'catalog',
 'duplicate_catalog_flag',
 'd

### NEA Example

In [5]:
from crossmatching import SimbadIdSupplier, NEACatalog

cm = Crossmatcher(NEACatalog(), SimbadIdSupplier())
cm.load_catalog(from_file="./pscomppars.txt")
cm.load_alternate_ids(input_table["star_name"], from_file="./alternate_ids_hpic.txt")
nea_result = cm.combined_crossmatch(input_table, input_starname_key="star_name")

In [6]:
Counter(nea_result["match_type"])

Counter({np.str_('id+coordinates'): 792,
         np.str_('coordinates'): 53,
         np.str_('id'): 15})

In [7]:
nea_result.colnames

['star_name',
 'sy_dist_input',
 'st_spectype_input',
 'st_rad_input',
 'st_teff_input',
 'st_mass_input',
 'st_age_input',
 'ra_input',
 'dec_input',
 'sy_vmag_input',
 'sy_jmag_input',
 'sy_hmag_input',
 'sy_kmag_input',
 'known_binary_fl',
 'gaia_binary_fl',
 'WDSsep',
 'wds_deltamag',
 'objectid',
 'pl_name',
 'pl_letter',
 'hostid',
 'hostname',
 'hd_name',
 'hip_name',
 'tic_id',
 'disc_pubdate',
 'disc_year',
 'disc_method',
 'discoverymethod',
 'disc_locale',
 'disc_facility',
 'disc_instrument',
 'disc_telescope',
 'disc_refname',
 'ra',
 'raerr1',
 'raerr2',
 'rasymerr',
 'rastr',
 'ra_solnid',
 'ra_reflink',
 'dec',
 'decerr1',
 'decerr2',
 'decsymerr',
 'decstr',
 'dec_solnid',
 'dec_reflink',
 'glon',
 'glonerr1',
 'glonerr2',
 'glonsymerr',
 'glonstr',
 'glon_solnid',
 'glon_reflink',
 'glat',
 'glaterr1',
 'glaterr2',
 'glatsymerr',
 'glatstr',
 'glat_solnid',
 'glat_reflink',
 'elon',
 'elonerr1',
 'elonerr2',
 'elonsymerr',
 'elonstr',
 'elon_solnid',
 'elon_reflink',


### Other Catalogs

In [8]:
import pyvo
import numpy as np
from astropy.table import Table, MaskedColumn
from crossmatching.catalogs.base import CatalogBase


class MyTAPCatalog(CatalogBase):
    ra_key = "ra"
    dec_key = "dec"
    hostname_key = "host"       
    planet_uid = "planet_name" 
    pm_key = None               
    pmerr_key = None            

    TAP_URL = "http://my-tap-service.example.org/tap"
    QUERY   = "SELECT * FROM my_schema.my_planets"

    def download(self) -> Table:
        service = pyvo.dal.TAPService(self.TAP_URL)
        return service.search(self.QUERY).to_table()

    def preprocess(self, table: Table) -> Table:
        return table

### Data "enrichment"
Computing parameters loosely

In [9]:
from crossmatching.enrichment import ParamFiller
from crossmatching.enrichment import NeaParamSource, EpicParamSource, ToiParamSource, SimbadParamSource, HpicParamSource
import glob
import numpy as np

nea_src = NeaParamSource()
nea_src.load(from_file='./pscomppars.txt', format='ascii')

epic_src = EpicParamSource()
epic_path = sorted(glob.glob('../Exo-MerCat/InputSources/epic_init*.csv'))[-1]
epic_src.load(from_file=epic_path, format='ascii.csv')

toi_src = ToiParamSource()
toi_path = sorted(glob.glob('../Exo-MerCat/InputSources/toi_init*.csv'))[-1]
toi_src.load(from_file=toi_path, format='ascii.csv')

full_emc = cme.catalog_table
nasa_names = [str(n).strip() for n in full_emc['nasa_name']]
nea_keys = set(nea_src._lookup)
has_nea = np.array([n in nea_keys for n in nasa_names])

simbad_src = SimbadParamSource()
simbad_src.load(from_file='simbad_params.txt')

hpic_src = HpicParamSource(emc_result)
hpic_src.load()

merger = ParamFiller([hpic_src, nea_src, epic_src, toi_src, simbad_src])
enriched = merger.enrich(full_emc)


TypeError: ParamFiller.enrich() missing 16 required positional arguments: 'planet_radius_key', 'planet_flux_key', 'planet_equilibrium_temperature_key', 'semi_major_axis_key', 'period_key', 'msini_key', 'star_spectral_type_key', 'star_radius_key', 'star_mass_key', 'star_effective_temperature_key', 'star_logg_key', 'star_metallicity_key', 'star_luminosity_key', 'vmag_key', 'kmag_key', and 'distance_key'

In [ ]:
set(enriched.colnames) - set(emc_result.colnames)

{'a_src',
 'flux_rel',
 'flux_rel_err1',
 'flux_rel_err2',
 'flux_rel_src',
 'pl_eqt',
 'pl_eqt_err1',
 'pl_eqt_err2',
 'pl_eqt_src',
 'r_earth',
 'r_earth_max',
 'r_earth_min',
 'r_earth_src',
 'spectral_category',
 'st_logg',
 'st_logg_err1',
 'st_logg_err2',
 'st_logg_src',
 'st_lum',
 'st_lum_err1',
 'st_lum_err2',
 'st_lum_src',
 'st_mass_err1',
 'st_mass_err2',
 'st_mass_src',
 'st_met',
 'st_met_err1',
 'st_met_err2',
 'st_met_src',
 'st_rad_err1',
 'st_rad_err2',
 'st_rad_src',
 'st_teff_err1',
 'st_teff_err2',
 'st_teff_src',
 'sy_dist_err1',
 'sy_dist_err2',
 'sy_dist_src',
 'sy_vmag_err1',
 'sy_vmag_err2',
 'sy_vmag_src'}

In [ ]:
from astropy.table import Table
from crossmatching import Crossmatcher, NEACatalog, SimbadIdSupplier

# Load input stellar survey
input_table = Table.read("input/HPIC_LC4_combined_d50.txt", format="ascii")

# Build the crossmatcher
cm = Crossmatcher(
    catalog=NEACatalog(),
    id_supplier=SimbadIdSupplier(),
)

# Load catalogs and alternate ids

# this line only needs to run once, or when newer catalog data is wanted
cm.catalog.save_raw("pscomppars1.txt") 
cm.load_catalog(from_file="pscomppars1.txt")


# this line only needs to run once, or when catalog ids changed
cm.id_supplier.save_raw(input_table["star_name"],"alternate_ids_hpic1.txt")
cm.load_alternate_ids(input_table["star_name"], from_file="alternate_ids_hpic1.txt")


# Run the combined crossmatch (ID-based first, then coordinate-based)
result = cm.combined_crossmatch(input_table, input_starname_key="star_name", input_epoch=2000)


from collections import Counter
print(Counter(result["match_type"])) # see the frequency of the used matching methods

result["star_name", "pl_name", "match_type", "crossmatching_angular_separation"]
# subtable is returned as cell output, alternatively use .show_in_notebook() or .show_in_browser() 

Counter({np.str_('id'): 807, np.str_('coordinates'): 51})


star_name,pl_name,match_type,crossmatching_angular_separation
,,,arcsec
str29,str29,str11,float64
TIC 245873777,alf Tau b,id,0.44218752885619694
TIC 423088367,HD 62509 b,id,0.4526870268041658
TIC 229540730,bet UMi b,id,0.009958822419128472
TIC 306349516,alf Ari b,id,0.5070403637119875
TIC 471015557,GJ 887 d,id,106.94598297752367
TIC 471015557,GJ 887 e,id,106.94598297752367
TIC 471015557,GJ 887 b,id,106.94598297752367
TIC 471015557,GJ 887 c,id,106.94598297752367


input_ids,id
str29,str33
TIC 459832522,TIC 459832522
TIC 459832522,AP J14153968+1910558
TIC 459832522,GALAH 150210005801171
TIC 459832522,GJ 541
TIC 459832522,HIP 69673
TIC 459832522,PLX 3242
TIC 459832522,LSPM J1415+1910
TIC 459832522,NLTT 36756
TIC 459832522,TYC 1472-1436-1


In [ ]:
# Example usage of the catalog ENRICH_KEYS constants
from crossmatching import NEACatalog, EMCCatalog
from crossmatching.enrichment import ParamFiller

# 1. Demonstrate enrichment of the NEA crossmatch table using NEACatalog.ENRICH_KEYS
filler = ParamFiller([hpic_src, nea_src, epic_src, toi_src, simbad_src])
enriched_nea = filler.enrich(nea_result, **NEACatalog.ENRICH_KEYS)
print("NEA Enriched columns:\n", [col for col in enriched_nea.colnames if not col.endswith('_src') and not col.endswith('_err1') and not col.endswith('_err2')])

# 2. Similarly, EMCCatalog.ENRICH_KEYS can configure planetary columns for Exo-MerCat tables:
# enriched_emc = filler.enrich(emc_result, **EMCCatalog.ENRICH_KEYS)
print("\nNEA ENRICH_KEYS dictionary:", NEACatalog.ENRICH_KEYS)
print("EMC ENRICH_KEYS dictionary:", EMCCatalog.ENRICH_KEYS)
